In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import lead, col, broadcast, collect_set, array_contains, size, datediff, avg, when, lit

class Transform:
    def __init__(self):
        pass

    def transform(self, inputDF):
        pass

class AirpodsAfterIphoneTransformer(Transform):

    def transform(self, inputDF):
        """
        Customers who bought Airpods after buying iPhone
        """
        
        transactionInputDF = inputDF.get("transactionInputDF")

        print("transactionInputDF in transform")
        
        transactionInputDF.show()

        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")

        transformedDF = transactionInputDF.withColumn(
            "next_Product_name", lead("product_name").over(windowSpec)
        )

        print("Customers who bought Airpods after iPhone")
        transformedDF.orderBy("customer_id", "transaction_date", "product_name").show()

        filteredDF = transformedDF.filter(
            (col("product_name") == "iPhone") & (col("next_product_name") == "AirPods")
            )
        
        filteredDF.orderBy("customer_id", "transaction_date", "product_name").show()

        customerInputDF = inputDF.get("customerInputDF")

        customerInputDF.show()

        joinDF = filteredDF.join(
            customerInputDF,
            "customer_id"
        )

        broadcastJoinDF = customerInputDF.join(
            broadcast(filteredDF),
            "customer_id"
        )

        print("Joined DF that shows the 1st Transformation")
        joinDF.show()

        print("Broadcast Join DF")
        broadcastJoinDF.show()

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )

class OnlyAirpodsAndIphone(Transform):

    def transform(self, inputDF):
        """
        Customer who have bought only iPhone and Airpods nothing else
        """

        transactionInputDF = inputDF.get("transactionInputDF")

        print("transactionInputDF in transform")
        
        transactionInputDF.show()

        groupedDF = transactionInputDF.groupBy("customer_id").agg(
            collect_set("product_name").alias("products")
        )

        print("Grouped DF")
        groupedDF.show()

        filteredDF = groupedDF.filter(
            (array_contains(col("products"), "iPhone")) &
            (array_contains(col("products"), "AirPods")) & 
            (size(col("products")) == 2)
        )
        
        print("Only Airpods and iPhone")
        filteredDF.show()

        customerInputDF = inputDF.get("customerInputDF")

        customerInputDF.show()

        joinDF =  filteredDF.join(
           broadcast(customerInputDF),
            "customer_id"
        )

        print("Joined DF that shows the 2nd Transformation")
        joinDF.show()

        return joinDF.select(
            "customer_id",
            "customer_name",
            "location"
        )

class AvgTimeIphoneToAirpodsTransformer(Transform):
    def transform(self, inputDF):
        """
        Calculate average time between iPhone and AirPods purchase
        """
        transactionInputDF = inputDF.get("transactionInputDF")
        customerInputDF = inputDF.get("customerInputDF")
        
        # Convert transaction_date to date type if it's not already
        transactionInputDF = transactionInputDF.withColumn(
            "transaction_date", 
            col("transaction_date").cast("date")
        )
        
        # Create window spec to identify next product purchase
        windowSpec = Window.partitionBy("customer_id").orderBy("transaction_date")
        
        # Add columns for next product and its purchase date
        purchaseSequenceDF = transactionInputDF.withColumn(
            "next_product_name", lead("product_name").over(windowSpec)
        ).withColumn(
            "next_transaction_date", lead("transaction_date").over(windowSpec)
        )
        
        # Filter for iPhone followed by AirPods purchases
        iphoneToAirpodsDF = purchaseSequenceDF.filter(
            (col("product_name") == "iPhone") & 
            (col("next_product_name") == "AirPods")
        )
        
        # Calculate days between purchases
        daysBetweenPurchasesDF = iphoneToAirpodsDF.withColumn(
            "days_between_purchases", 
            datediff(col("next_transaction_date"), col("transaction_date"))
        )
        
        # Join with customer data
        resultDF = daysBetweenPurchasesDF.join(
            broadcast(customerInputDF),
            "customer_id"
        )
        
        # Calculate average time between purchases by customer and overall
        customerAvgDF = resultDF.groupBy("customer_id", "customer_name", "location").agg(
            avg("days_between_purchases").alias("avg_days_to_airpods")
        )
        
        # Calculate overall average
        overallAvg = resultDF.agg(
            avg("days_between_purchases").alias("overall_avg_days")
        ).collect()[0]["overall_avg_days"]
        
        # Add overall average as a column
        finalDF = customerAvgDF.withColumn(
            "overall_avg_days", 
            lit(overallAvg)
        )
        
        return finalDF
